# Video Interview Base Generation Pipeline

**Steps:**
1. Run WhisperX transcription + speaker diarization
2. Parse results with `audio_sep.py` → `diarize_segments.csv`, `diarize_result.json`, `script_timeline.csv`
3. Identify narrator (speaker with longest total text)
4. Extract interview segments using `construct_interview_base.py`

**Outputs:**
- `interview.json` — structured segments with speaker, text, timestamps
- `interview.txt` — plain text utterances
- `interview_visuals_random.npy` — visual embeddings (optional, shape: [N, 2304])

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q git+https://github.com/m-bain/whisperX.git
!pip install -q pydub librosa soundfile
print('Done')

In [ ]:
import json
import math
import os
import pathlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import whisperx
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## 2. Configuration

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
VIDEO_PATH    = '/content/drive/MyDrive/your_video.mp4'  # path to input video
VIDEO_NAME    = 'my_video'                               # short id, no spaces
OUTPUT_DIR    = Path('/content/drive/MyDrive/interview_base_output')
HF_TOKEN      = 'hf_YOUR_TOKEN_HERE'                    # HuggingFace token for diarization

# WhisperX
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
CUDA_INDEX    = 0
MODEL_CONFIG  = 'large-v2'   # tiny | base | small | medium | large-v2
BATCH_SIZE    = 8
COMPUTE_TYPE  = 'float16'    # float16 | int8
LANGUAGE      = 'en'

# Optional: path to pre-computed CLIP features {VIDEO_NAME}_clip.npy
VISUAL_EMBEDDING_PATH = None  # e.g. '/content/drive/MyDrive/my_video_clip.npy'
# ────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Internal paths (mirrors audio_sep.py structure)
VIDEO_DIR = OUTPUT_DIR / VIDEO_NAME[0] / VIDEO_NAME
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_DIR}')

## 3. Helper Functions (from audio_sep.py + construct_interview_base.py)

In [ ]:
# ── Helpers from audio_sep.py ────────────────────────────────────────────────

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def save_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def save_txt(path, lines):
    with open(path, 'w') as f:
        for line in lines:
            f.write(str(line) + '\n')


def group_by_speaker(transcript):
    """Group WhisperX segments by speaker id."""
    speakers = {}
    for entry in transcript:
        if 'speaker' not in entry:
            print('No speaker found in entry:', entry)
            continue
        speaker = entry['speaker']
        if speaker not in speakers:
            speakers[speaker] = []
        speakers[speaker].append((entry['start'], entry['end'], entry['text']))
    return speakers


def text_script(diarize_result, output_dir):
    """Write per-speaker .txt files (concatenated transcripts)."""
    for speaker, content_list in diarize_result.items():
        content_str = ''.join(str(seg[2]) for seg in content_list)
        output_path = output_dir / f'{speaker}.txt'
        with open(output_path, 'a') as f:
            f.write(content_str)
    print(f'Wrote {len(diarize_result)} speaker txt files to {output_dir}')


def narrator_selection(video_dir):
    """Identify narrator as the speaker whose .txt file has the most characters."""
    txt_files = list(video_dir.glob('*.txt'))
    longest_id = None
    longest_len = 0
    for txt_file in txt_files:
        content = txt_file.read_text()
        if len(content) > longest_len:
            longest_len = len(content)
            longest_id = txt_file.stem  # filename without extension = speaker id
    print(f'Narrator: {longest_id} (text length: {longest_len} chars)')
    return longest_id


def find_exact_matching_text(diarize_result, output_dir):
    """Convert grouped JSON to a time-sorted script_timeline.csv."""
    rows = []
    for speaker, segments in diarize_result.items():
        for segment in segments:
            start_time, end_time, text = segment
            text = text.strip().replace('"', '')
            rows.append([speaker, start_time, end_time, text])

    df = pd.DataFrame(rows, columns=['Speaker', 'Start Time', 'End Time', 'Text'])
    df_sorted = df.sort_values(by=['End Time'])
    output_path = output_dir / 'script_timeline.csv'
    df_sorted.to_csv(output_path, index=False)
    print(f'Saved script_timeline.csv: {len(df_sorted)} rows')
    return df_sorted


# ── Helpers from construct_interview_base.py ─────────────────────────────────

def construct_visual_embedding(clip_frame_embeddings):
    """Select 3 random frames from a segment and flatten to [1, 2304]."""
    num_visual = clip_frame_embeddings.shape[0]
    if num_visual >= 3:
        indices = torch.randperm(num_visual)[:3]
    else:
        indices = torch.randint(0, num_visual, (3,))
    selected = clip_frame_embeddings[indices]
    return selected.reshape(1, -1)


print('Helper functions loaded.')

## 4. Run WhisperX (diarize_audio)

In [ ]:
# Mirrors diarize_audio() from audio_sep.py
output_csv  = VIDEO_DIR / 'diarize_segments.csv'
output_json = VIDEO_DIR / 'diarize_result.json'

if output_csv.exists() and output_json.exists():
    print('Already processed — loading cached results.')
    diarize_result = load_json(output_json)
    diarize_segments = pd.read_csv(output_csv)
else:
    print('Loading WhisperX model...')
    model = whisperx.load_model(
        MODEL_CONFIG, DEVICE,
        device_index=CUDA_INDEX,
        compute_type=COMPUTE_TYPE
    )

    print('Loading audio...')
    audio = whisperx.load_audio(VIDEO_PATH)

    print('Transcribing...')
    result = model.transcribe(audio, batch_size=BATCH_SIZE)

    print('Aligning...')
    model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=DEVICE)
    result = whisperx.align(result['segments'], model_a, metadata, audio, DEVICE,
                            return_char_alignments=False)

    print('Running speaker diarization...')
    diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=DEVICE)
    diarize_segments = diarize_model(audio)

    result = whisperx.assign_word_speakers(diarize_segments, result)

    # Save diarize_segments.csv
    diarize_segments.to_csv(output_csv)

    # Group by speaker and save diarize_result.json
    diarize_result = group_by_speaker(result['segments'])
    save_json(output_json, diarize_result)

    print(f'Speakers found: {list(diarize_result.keys())}')

print(f'Total speakers: {len(diarize_result)}')
print(f'Total segments: {sum(len(v) for v in diarize_result.values())}')

## 5. Build script_timeline.csv + Per-Speaker .txt Files

In [ ]:
# text_script: write per-speaker .txt files
text_script(diarize_result, VIDEO_DIR)

# find_exact_matching_text: build time-sorted script_timeline.csv
timeline_df = find_exact_matching_text(diarize_result, VIDEO_DIR)

# Also save as script_timeline_combined.csv (expected by construct_interview_base.py)
combined_path = VIDEO_DIR / 'script_timeline_combined.csv'
timeline_df.to_csv(combined_path, index=False)
print(f'Saved script_timeline_combined.csv')

display(timeline_df.head(10))

## 6. Identify Narrator (narrator_selection)

In [ ]:
# narrator_selection: narrator = speaker with longest concatenated text
narrator_id = narrator_selection(VIDEO_DIR)

# Save narrator_id_by_length.csv
narrator_df = pd.DataFrame({'video_name': [VIDEO_NAME], 'narrator_id': [narrator_id]})
narrator_csv_path = OUTPUT_DIR / 'narrator_id_by_length.csv'
narrator_df.to_csv(narrator_csv_path, index=False)
print(f'Saved narrator_id_by_length.csv')

# Speaker duration summary
print('\nSpeaker summary:')
for speaker, segs in diarize_result.items():
    total_dur = sum(s[1] - s[0] for s in segs)
    label = ' ← NARRATOR' if speaker == narrator_id else ''
    print(f'  {speaker}: {len(segs)} segments, {total_dur:.1f}s{label}')

## 7. Extract Interview Segments (process_video_full)

In [ ]:
# process_video_full() from construct_interview_base.py
# Extracts all segments where Speaker != narrator_id  (non-narrator = interviews)

# Load visual embeddings if provided
visual_np = None
if VISUAL_EMBEDDING_PATH and Path(VISUAL_EMBEDDING_PATH).exists():
    visual_np = np.load(VISUAL_EMBEDDING_PATH)
    print(f'Loaded visual embeddings: {visual_np.shape}')
else:
    print('No visual embeddings — running text-only extraction.')

interview_segments = []
interview_texts    = []
interview_visuals  = []

for _, row in timeline_df.iterrows():
    if row['Speaker'] == narrator_id:
        continue  # skip narrator — keep only non-narrator (interview) segments

    segment = {
        'Speaker':    row['Speaker'],
        'Text':       row['Text'],
        'Start Time': row['Start Time'],
        'End Time':   row['End Time'],
        'part':       'main'
    }
    interview_segments.append(segment)
    interview_texts.append(row['Text'])

    if visual_np is not None:
        start_idx = max(math.floor(row['Start Time']) - 1, 0)
        end_idx   = min(math.ceil(row['End Time']), visual_np.shape[0])
        seg_vis   = visual_np[start_idx:end_idx]
        compressed = construct_visual_embedding(seg_vis)
        assert compressed.shape[0] == 1
        interview_visuals.append(compressed)

print(f'Interview segments: {len(interview_segments)}')
print(f'Unique speakers:   {len(set(s["Speaker"] for s in interview_segments))}')

## 8. Save Outputs

In [ ]:
# interview.json
json_path = VIDEO_DIR / 'interview.json'
save_json(json_path, interview_segments)
print(f'Saved {json_path}')

# interview.txt
txt_path = VIDEO_DIR / 'interview.txt'
save_txt(txt_path, interview_texts)
print(f'Saved {txt_path}')

# interview_visuals_random.npy (only if visual embeddings were provided)
if interview_visuals:
    visuals_stacked = np.vstack(interview_visuals)
    assert len(interview_segments) == visuals_stacked.shape[0]
    npy_path = VIDEO_DIR / 'interview_visuals_random.npy'
    np.save(npy_path, visuals_stacked)
    print(f'Saved {npy_path}  shape: {visuals_stacked.shape}')

print(f'\nAll outputs in: {VIDEO_DIR}')

## 9. Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Speaker duration bar chart
speaker_durations = {
    sp: sum(s[1] - s[0] for s in segs)
    for sp, segs in diarize_result.items()
}
speakers = list(speaker_durations.keys())
durations = list(speaker_durations.values())
colors = ['#e74c3c' if s == narrator_id else '#3498db' for s in speakers]

ax1.barh(speakers, durations, color=colors, alpha=0.8)
ax1.set_xlabel('Duration (seconds)')
ax1.set_title('Speaker Duration  (red = narrator)')

# Timeline
for i, sp in enumerate(speakers):
    color = '#e74c3c' if sp == narrator_id else '#3498db'
    for seg in diarize_result[sp]:
        ax2.barh(i, seg[1] - seg[0], left=seg[0], height=0.7, color=color, alpha=0.6)

ax2.set_yticks(range(len(speakers)))
ax2.set_yticklabels(speakers)
ax2.set_xlabel('Time (seconds)')
ax2.set_title('Speech Timeline')

plt.tight_layout()
fig.savefig(VIDEO_DIR / 'speaker_analysis.png', dpi=100)
plt.show()

# Summary
total = sum(durations)
interview_dur = sum(s['End Time'] - s['Start Time'] for s in interview_segments)
print(f'\nSummary')
print(f'  Total video duration : {total:.1f}s')
print(f'  Narrator ({narrator_id}): {speaker_durations[narrator_id]:.1f}s ({speaker_durations[narrator_id]/total*100:.0f}%)')
print(f'  Interview segments   : {len(interview_segments)} ({interview_dur:.1f}s, {interview_dur/total*100:.0f}%)')